# Artificial Intelligence
Group members
- Mtume Owino
- James Mathenge
- Joyce Mbiri


This project investigates whether an AI-driven mobile learning companion can generate accurate,
actionable competency feedback for CBC learners and reduce teacher assessment workload in Kenyan primary schools.


In [ ]:
RANDOM_STATE = 42
TARGET_COLUMN = "Exam_Score"
CATEGORICAL_FEATURES = [
    "Parental_Involvement",
    "Access_to_Resources",
    "Extracurricular_Activities",
    "Motivation_Level",
    "Internet_Access",
    "Family_Income",
    "Teacher_Quality",
    "School_Type",
    "Peer_Influence",
    "Learning_Disabilities",
    "Parental_Education_Level",
    "Distance_from_Home",
    "Gender",
]
NUMERICAL_FEATURES = [
    "Hours_Studied",       # weekly practice time
    "Attendance",          # class attendance rate
    "Sleep_Hours",         # average nightly sleep
    "Previous_Scores",     # running average of past assessments
    "Tutoring_Sessions",   # extra support received
    "Physical_Activity",   # weekly activity level
]
EXPECTED_COLUMNS = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
BAND_LABELS = [
    "Below Expectation",
    "Approaching Expectation",
    "Meeting Expectation",
    "Exceeding Expectation",
]

In [ ]:
import kagglehub
from pathlib import Path
import pandas as pd

KAGGLE_SLUG = "lainguyn123/student-performance-factors"
download_dir = Path(kagglehub.dataset_download(KAGGLE_SLUG))

# Dynamically search for the CSV file in the downloaded directory
csv_files = list(download_dir.glob("**/*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV file found in {download_dir}")

df = pd.read_csv(csv_files[0])
print(f"Loaded {len(df)} rows from {csv_files[0].name}.")

Using Colab cache for faster access to the 'student-performance-factors' dataset.
Loaded 6607 rows from StudentPerformanceFactors.csv.
0       3
1       4
2       4
3       4
4       4
       ..
6602    2
6603    2
6604    2
6605    3
6606    4
Name: Physical_Activity, Length: 6607, dtype: int64


### 1. Cleaning the data


In [ ]:
 # 1. A row without a target cannot be used for supervised learning.
missing_target = df[TARGET_COLUMN].isna()
df = df.loc[missing_target].copy()
print(f"Rows with missing data: {int(missing_target.sum())}")


# 2. Drop duplicate records.
before = len(df)
df = df.drop_duplicates().copy()
print(f"Duplicate rows removed: {int(before - len(df))}")


# 3. Exam scores above 100 are invalid
invalid_scores = int((df[TARGET_COLUMN] > 100).sum())
df[TARGET_COLUMN] = df[TARGET_COLUMN].clip(lower=0, upper=100)
print(f"Invalid scores {invalid_scores}")

# 4. Attendance cannot exceed 100% either.
invalid_attendance = int((df["Attendance"] > 100).sum())
df["Attendance"] = df["Attendance"].clip(lower=0, upper=100)
print(f"Number Invalid attendance values: {invalid_attendance}")


# 5. Standardise category spelling so one-hot columns stay stable.
for column in CATEGORICAL_FEATURES:
    df[column] = df[column].str.strip().str.title()

Rows with missing data: 0
Duplicate rows removed: 0
Invalid scores 0
Number Invalid attendance values: 0


# 2. Buidling the scores

In [ ]:
import numpy as np

quantiles = df[TARGET_COLUMN].quantile([0.25, 0.50, 0.75]).to_numpy() # Exam_Score converted into quartiles
if len(np.unique(quantiles)) < 3:
    raise ValueError(
        "Exam_Score has too many tied values to build four quartile bands."
    )
cut_points = [float(q) for q in quantiles]

lower, middle_low, middle_high = cut_points # [65., 67., 69.]
bins = [-np.inf, lower, middle_low, middle_high, np.inf] # [-inf, .65, .67, .69, inf]

df["CBC_Band"] = pd.cut(df[TARGET_COLUMN], bins=bins, labels=BAND_LABELS) # takes each Exam_Score and checks which interval it falls into.
df = df.dropna(subset=["CBC_Band"]).copy()
df["CBC_Band"] = df["CBC_Band"].astype(str) # because pd.cut returns a special column

print(f"Quartiles: {[float(p) for p in cut_points]}")

Quartiles: [65.0, 67.0, 69.0]


### 3. Training the model

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold


def build_preprocessor() -> ColumnTransformer:
    """
    Shared preprocessing, reused by every model.

    All preprocessing is inside the pipeline so cross-validation refits it on
    each fold. That is what prevents data leakage.
    """


    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")), # Missing numeric values are replaced with the median of that column
            ("scaler", StandardScaler()), # Convert each numeric column to a standard scale (value - mean) / standard deviation
        ]
    )


    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")), # Fill in missing categories with the most common category in that column
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)), # Convert text categories to numbers
        ]
    )


    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, NUMERICAL_FEATURES),
            ("cat", categorical_transformer, CATEGORICAL_FEATURES),
        ]
    )


def build_pipeline() -> Pipeline:
    """Full preprocessing + XGBoost classifier pipeline."""
    classifier = XGBClassifier(
        n_estimators=120, # 120 trees
        max_depth=5, # control complexity
        learning_rate=0.08, # each tree makes a small correction; smaller values usually generalise better.
        random_state=RANDOM_STATE, # makes training reproducable
        eval_metric="mlogloss",
    ) # XGBoost object

    return Pipeline(
        [("preprocessor", build_preprocessor()), ("classifier", classifier)]
    )

X = df[EXPECTED_COLUMNS] # Feature matrix
y = df["CBC_Band"]
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y) # converts the four classes to integers 0–3.
print(f"\nEncoder classes: {list(label_encoder.classes_)}")

X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=0.2, # 20% test, 80% train
        random_state=RANDOM_STATE, # For reproducability on every run
        stratify=y_encoded, # each run gets an equal amount of bands
) # Splits the data into four pieces: training features, test features, training labels, test labels.

pipeline = build_pipeline()

# Cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)# shuffle before splitting
cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"}, # Compute two metrics per fold: accuracy and macro F1.
        n_jobs=1, # Run sequentially
)

print(f"CV accuracy: {cv_results["test_accuracy"]}")
print(f"CV F1: {cv_results["test_f1_macro"]}")




Encoder classes: ['Approaching Expectation', 'Below Expectation', 'Exceeding Expectation', 'Meeting Expectation']
CV accuracy: [0.73793756 0.7473983  0.75402081 0.73509934 0.73320719]
CV F1: [0.71622737 0.72454154 0.73626512 0.7116628  0.70913582]


# Validation

To validate our model we use a confusion matrix and f1 macro that inform the 5 fold CV

Using 1322 held out students.


##### Confusion matrix
Confusion matrix allows us to see what was correctly and wrongly guessed for each competency band in the dataset

| Truth ↓ / Predicted → | Below | Approaching | Meeting | Exceeding | Total |
|---|---:|---:|---:|---:|---:|
| **Below** | **381** | 41 | 4 | 0 | 426 |
| **Approaching** | 57 | **176** | 58 | 3 | 294 |
| **Meeting** | 0 | 78 | **152** | 47 | 277 |
| **Exceeding** | 5 | 4 | 44 | **272** | 325 |

The total number of mistakes is everything off the diagonal of the confusion matrix
45 + 118 + 125 + 53 = 341
or 1322 - 918 = 341

341 wrong / 1,322 learners = 0.258
981 correct / 1,322 learners = 0.742





##### F1
Precision = when the model said Approaching, how often was it right?
Recall = of the learners who really are Approaching, how many did it find?
For Approaching: 2 × (0.59 × 0.60) / (0.59 + 0.60) = 0.59

F1 = 2 × (precision × recall) / (precision + recall)


| Band | Precision | Recall | F1 | Learners |
|---|---:|---:|---:|---:|
| Below | 0.86 | 0.89 | 0.88 | 426 |
| Approaching | 0.59 | 0.60 | 0.59 | 294 |
| Meeting | 0.59 | 0.55 | 0.57 | 277 |
| Exceeding | 0.84 | 0.84 | 0.84 | 325 |


##### Macro F1
Same as F1 but using average
Accuracy counts every learner equally, so the largest band dominates and a weak middle band can hide. Macro F1 counts every band equally.

macro F1 = (0.88 + 0.59 + 0.57 + 0.84) / 4 = 0.72



One split cannot be tested so we used a
Stratified 5 fold Cross validation
Each round gets a fair share of each band (stratified)
Out of 6607 learners 1321 (20%) are used for testing and 5286 (80%) are used for training


| Fold | 1 | 2 | 3 | 4 | 5 |
|---|---:|---:|---:|---:|---:|
| Accuracy | 0.741 | 0.729 | 0.761 | 0.768 | 0.734 |


```
mean = (0.741 + 0.729 + 0.761 + 0.768 + 0.734) / 5 = 0.747
```

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

def build_models() -> dict[str, object]:
    """The four classifiers, all with the same preprocessing."""
    return {
        #"Logistic Regression": LogisticRegression(max_iter=2000),
        "XGBoost": XGBClassifier(
            n_estimators=120,
            max_depth=5,
            learning_rate=0.08,
            random_state=RANDOM_STATE,
            eval_metric="mlogloss",
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    }



for name, classifier in build_models().items():
  pipeline = Pipeline(
            [("preprocessor", build_preprocessor()), ("classifier", classifier)]
  )
  cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
  scores = cross_validate(
            pipeline, X, y_encoded, cv=cv, scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"}
            , n_jobs=1
  )

  pipeline.fit(X_train, y_train)
  predictions = pipeline.predict(X_test)
  cv_results = cross_validate(
          pipeline,
          X_train,
          y_train,
          cv=cv,
          scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"},
          n_jobs=1,
  )

  print(f"{name}:")
  print(f"CV accuracy: {cv_results["test_accuracy"]}")
  print(f"CV F1: {cv_results["test_f1_macro"]}")
  print('\n')


XGBoost CV accuracy: [0.73793756 0.7473983  0.75402081 0.73509934 0.73320719]
XGBoost CV F1: [0.71622737 0.72454154 0.73626512 0.7116628  0.70913582]
Random Forest CV accuracy: [0.70955535 0.69536424 0.7038789  0.69820246 0.68779565]
Random Forest CV F1: [0.66730797 0.64992153 0.66455958 0.65859718 0.64857819]
Decision Tree CV accuracy: [0.59697256 0.57899716 0.58088931 0.61778619 0.60927152]
Decision Tree CV F1: [0.57602228 0.5628549  0.56159171 0.59703717 0.58468773]


# Data set Observations

-  Exam_Score is narrow and targeted has mean 67.2, standard deviation 3.9, minimum 55, maximum 101. Q1 = 65, median = 67, Q3 = 69. Half of all learners fall inside a 4-point range (65–69).


- Attendance shows the strongest correlation to Exam score


- Hours studied shows the second strongest correltaion to Exam score

- Missing data is limited and concentrated

In [ ]:
# Exam_Score is narrow and targeted
print("Exam_Score is narrow and targeted")
score = df["Exam_Score"]
print(score.describe().round(2))
print("\n")



# Attendance shows the strongest correlation to Exam score
print("Attendance shows the strongest correlation to Exam score")
print(df["Attendance"].corr(df["Exam_Score"]))
print("\n")

# Hours studied shows the second strongest correltaion to Exam score
print("Hours studied shows the second strongest correltaion to Exam score")
print(round(df["Hours_Studied"].corr(df["Exam_Score"]), 3))
print("\n")

# Missing data is limited and concentrated
print("Missing data is limited and concentrated ")
missing = df.isna().sum()
print("Total missing cells:", int(missing.sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("\n")







Exam_Score is narrow and targeted
count    6607.00
mean       67.24
std         3.89
min        55.00
25%        65.00
50%        67.00
75%        69.00
max       100.00
Name: Exam_Score, dtype: float64


Attendance shows the strongest correlation to Exam score
0.5812045285301284


Hours studied shows the second strongest correltaion to Exam score
0.446


Missing data is limited and concentrated 
Total missing cells: 235
Duplicate rows: 0


